# Semana 01: Arquitetura TI/TA, Pirâmide ISA-95 e Taxonomia dos Sinais Industriais

## Módulo de Arquitetura e Sinais Fabris — Fábrica Virtual Smart N1

Este notebook apresenta a arquitetura de referência da **norma ANSI/ISA-95 (IEC 62264)** e a taxonomia de **sinais industriais (Discretos, Contínuos e PWM)** sob a perspectiva da **Tecnologia da Informação (TI) e Engenharia de Software**, transformando eventos do chão de fábrica em dados estruturados.

### Objetivos de aprendizagem
- Compreender a evolução histórica das revoluções industriais até os Sistemas Ciber-Físicos (CPS).
- Dominar a arquitetura da norma **ANSI/ISA-95 (IEC 62264)** e o detalhamento dos seus 5 níveis hierárquicos.
- Analisar os 3 tipos fundamentais de sinais industriais: Discreto (0/1), Contínuo (Analógico 0-10V / 4-20mA) e PWM (Modulação por Largura de Pulso).
- Compreender a conversão de grandezas físicas em variáveis de software (Boolean, Float, Integer, Duty Cycle %).
- Executar scripts Python para amostragem de sinais, conversão ADC de 12 bits, cálculo PWM e geração de payloads JSON.

---


## 1. Fundamentação teórica

### 1.1 A Norma ANSI/ISA-95 e a Pirâmide da Automação Industrial

A norma **ANSI/ISA-95** (adotada internacionalmente como **IEC 62264**) estabelece o modelo de referência para a integração entre os sistemas de controle no chão de fábrica e os sistemas corporativos de gestão de TI.

![Pirâmide ISA-95](img/piramide_isa95.jpg)

#### Detalhamento dos Níveis da Pirâmide ISA-95:

1. **Nível 0: Processo Físico (Chão de Fábrica):**
   - *Função:* Interface física direta com a matéria-prima e equipamentos.
   - *Dispositivos:* Sensores discretos (indutivos, fotoelétricos), sensores analógicos (pressão, temperatura), atuadores pneumáticos, válvulas solenoide, motores e servomotores.
   - *Escala Temporal:* Resposta em tempo real contínuo (microssegundos a milissegundos).

2. **Nível 1: Controle Discreto e Contínuo:**
   - *Função:* Execução do algoritmo de controle determinístico, intertravamentos de segurança e malhas de regulação.
   - *Dispositivos:* Controladores Lógicos Programáveis (CLPs), PACs (Programmable Automation Controllers), SDCDs.
   - *Escala Temporal:* Ciclo de varredura (SCAN) entre 1 ms e 50 ms.

3. **Nível 2: Supervisão e Controle de Processo:**
   - *Função:* Monitoramento visual da planta em tempo real, operação de linhas, gestão de alarmes e registro histórico (Historian).
   - *Sistemas:* SCADA (Supervisory Control and Data Acquisition), IHMs (Interface Homem-Máquina).
   - *Escala Temporal:* Atualização gráfica entre 100 ms e 2 s.

4. **Nível 3: Gestão das Operações de Manufatura (MOM / MES):**
   - *Função:* Sequenciamento fino de ordens de produção, rastreabilidade de lotes (genealogia do produto), controle de qualidade e cálculo do índice OEE (Overall Equipment Effectiveness).
   - *Sistemas:* MES (Manufacturing Execution System), LIMS (Laboratory Information Management System).
   - *Escala Temporal:* Horas, turnos de trabalho, dias.

5. **Nível 4: Planejamento Empresarial e Logística:**
   - *Função:* Gestão financeira, compras de matéria-prima, vendas, logística global e planejamento estratégico.
   - *Sistemas:* ERP (Enterprise Resource Planning - ex: SAP, TOTVS), CRM, SCM.
   - *Escala Temporal:* Semanas, meses, anos.


### 1.2 Convergência TI/TA e Taxonomia dos Sinais Industriais

![Taxonomia dos Sinais Industriais sob a ótica de TI](img/sinais_industriais_ti.jpg)

Na perspectiva da TI, os dispositivos do Nível 0 e Nível 1 emitem 3 categorias fundamentais de sinais elétricos que são convertidos para variáveis de software:

#### 1. Sinal Discreto (Digital Binário: 0/1)
- **Fenômeno Físico:** Presença de peça, contato mecânico acionado, nível máximo atingido.
- **Representação Elétrica:** Tensão $0\text{V DC}$ (Falso) ou $+24\text{V DC}$ (Verdadeiro).
- **Tipo em Software:** `bool` (`True` / `False`).
- **Exemplo:** Sensor indutivo detectando a chegada de uma caixa metálica.

#### 2. Sinal Contínuo (Analógico: 0-10V / 4-20mA)
- **Fenômeno Físico:** Variação contínua de temperatura, pressão ou vazão.
- **Representação Elétrica:** Sinal proporcional em tensão ($0\text{ a }10\text{V}$) ou corrente ($4\text{ a }20\text{mA}$).
- **Processamento em TI:** Amostragem por um **Conversor Analógico-Digital (ADC)** de $12\text{ bits}$ ($0\text{ a }4095$) ou $16\text{ bits}$ ($0\text{ a }65535$).
- **Tipo em Software:** `float` (ex: `temperatura_celsius = 42.8`).

#### 3. Sinal PWM (Pulse Width Modulation — Modulação por Largura de Pulso)
- **Conceito:** Pulso digital quadrado onde a largura do tempo ativo ($t_{on}$) varia em relação ao período total ($T$).
- **Duty Cycle ($D$):**

$$D = \frac{t_{on}}{T} \times 100\%$$

- **Tensão Média Equivalente ($V_{avg}$):**

$$V_{avg} = D \times V_{max}$$

- **Para que serve em TI e Automação?**
  1. *Controle de Potência:* Controle de velocidade de motores e temperatura de resistências sem dissipação ôhmica em resistores.
  2. *Emulação de Saída Analógica:* Permite que microcontroladores e CLPs gerem tensões médias contínuas usando apenas saídas digitais em alta velocidade, dispensando conversores DAC caros.
- **Tipo em Software:** Percentual `float` ($0.0\%$ a $100.0\%$).


### 1.3 Quadro Comparativo TI vs TA

| Característica | Tecnologia da Automação (TA / OT) | Tecnologia da Informação (TI / IT) |
| :--- | :--- | :--- |
| **Foco Principal** | Disponibilidade física e segurança do processo (Safety) | Confidencialidade e integridade dos dados |
| **Determinismo** | Crítico em Tempo Real (latência imprevisível pode causar acidentes) | Latência tolerável (requisições HTTP, relatórios) |
| **Prioridade de Segurança** | **AIC:** Availability (1º), Integrity (2º), Confidentiality (3º) | **CIA:** Confidentiality (1º), Integrity (2º), Availability (3º) |
| **Ciclo de Vida** | 10 a 20 anos (CLPs, motores, sensores) | 3 a 5 anos (Servidores, notebooks, frameworks) |
| **Protocolos** | Modbus, Profinet, OPC UA, MQTT | HTTP/HTTPS, REST APIs, WebSockets, SQL |

---


## 2. Arquitetura da atividade

Nesta prática, construímos o pipeline completo de conversão e estruturação de dados entre o chão de fábrica e os sistemas de software de TI:

### Detalhamento do Pipeline de Dados:

1. **Captura Física de Sinais (Nível 0 - Campo):**
   - Coleta simultânea do estado de sensores discretos de presença (0V/24V), leitor de temperatura analógico em malha de 12-bits e gerador de pulsos de velocidade PWM.

2. **Processamento e Amostragem (Nível 1 - Controlador):**
   - Validação de limiares lógicos para sinais discretos.
   - Conversão de contagens brutas do ADC de 12-bits (0 a 4095) para a unidade de engenharia real ($^\circ\text{C}$).
   - Cálculo do *Duty Cycle* (%) e tensão média equivalente ($V_{avg}$) da saída PWM de controle do motor.

3. **Serialização e Entrega para TI (Níveis 2/3 - MES/Cloud):**
   - Empacotamento das variáveis de processo em uma mensagem estruturada no formato **JSON**, contendo identificador de estacão, timestamp ISO-8601 e métricas prontas para consumo por brokers MQTT e bancos de dados.

**Esquema Resumido do Pipeline:**

**Sinal Elétrico de Campo (Discreto / ADC 12-bits / PWM) → Algoritmo Decodificador (Python) → Payload JSON Estruturado de Telemetria**

---


## 3. Prática — Interpretador de Sinais Fabris e Conversores em Python

### Passo 1 — Simular a Decodificação dos 3 Tipos de Sinais
Execute o script abaixo para simular como o software de TI interpreta sinais discretos, converte leituras analógicas de ADC 12-bits e calcula a tensão média equivalente de um sinal PWM.

In [1]:
import json
import time
import random

def ler_sinal_discreto(tensao_volts):
    estado_logico = tensao_volts >= 18.0
    return {"tipo": "DISCRETO", "tensao_v": tensao_volts, "valor_logico": estado_logico}

def converter_sinal_analogico_adc(valor_adc_12bit, min_eng=0.0, max_eng=100.0):
    percentual = valor_adc_12bit / 4095.0
    valor_engenharia = min_eng + (percentual * (max_eng - min_eng))
    return {"tipo": "ANALÓGICO_ADC", "raw_adc": valor_adc_12bit, "valor_medido": round(valor_engenharia, 2)}

def decodificar_sinal_pwm(tempo_ativo_ms, periodo_total_ms, tensao_max_v=24.0):
    duty_cycle_pct = (tempo_ativo_ms / periodo_total_ms) * 100.0
    tensao_media_v = (duty_cycle_pct / 100.0) * tensao_max_v
    return {"tipo": "PWM", "duty_cycle_pct": round(duty_cycle_pct, 1), "tensao_media_v": round(tensao_media_v, 2)}

# Exemplo de amostragem no software
s_discreto = ler_sinal_discreto(24.0)
s_analogico = converter_sinal_analogico_adc(3072, min_eng=0.0, max_eng=100.0)
s_pwm = decodificar_sinal_pwm(7.5, 10.0, tensao_max_v=24.0)

print("=== SIMULAÇÃO DE DECODIFICAÇÃO DE SINAIS PELO SOFTWARE DE TI ===\n")
print("1. Sinal Discreto: ", s_discreto)
print("2. Sinal Analógico:", s_analogico)
print("3. Sinal PWM:      ", s_pwm)


=== SIMULAÇÃO DE DECODIFICAÇÃO DE SINAIS PELO SOFTWARE DE TI ===

1. Sinal Discreto:  {'tipo': 'DISCRETO', 'tensao_v': 24.0, 'valor_logico': True}
2. Sinal Analógico: {'tipo': 'ANALÓGICO_ADC', 'raw_adc': 3072, 'valor_medido': 75.02}
3. Sinal PWM:       {'tipo': 'PWM', 'duty_cycle_pct': 75.0, 'tensao_media_v': 18.0}


### Passo 2 — Empacotar as Leituras em Payload JSON de Telemetria
Rode a célula para gerar o payload unificado consumido por APIs REST e brokers MQTT.

In [2]:
payload_ti = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "estacao_id": "SMART_N1_CELULA_01",
    "sensores": {
        "presenca_peca_bool": s_discreto["valor_logico"],
        "temperatura_processo_celsius": s_analogico["valor_medido"],
        "comando_velocidade_pwm_pct": s_pwm["duty_cycle_pct"],
        "tensao_media_motor_volts": s_pwm["tensao_media_v"]
    }
}

print("=== PAYLOAD JSON UNIFICADO PARA SISTEMAS DE TI ===")
print(json.dumps(payload_ti, indent=2))


=== PAYLOAD JSON UNIFICADO PARA SISTEMAS DE TI ===
{
  "timestamp": "2026-08-18T19:19:20Z",
  "estacao_id": "SMART_N1_CELULA_01",
  "sensores": {
    "presenca_peca_bool": true,
    "temperatura_processo_celsius": 75.02,
    "comando_velocidade_pwm_pct": 75.0,
    "tensao_media_motor_volts": 18.0
  }
}


---

## 4. Exercícios de fixação e avaliação

### Questão 1
Classifique cada um dos sistemas abaixo no respectivo nível da Pirâmide ISA-95 (0 a 4):
- a) Sensor fotoelétrico e atuador solenoide na esteira.
- b) Controlador Lógico Programável (CLP) executando malha PID.
- c) Sistema SCADA exibindo telas de sinótico da planta.
- d) Sistema MES rastreando ordens de produção e lote.
- e) ERP gerenciando compras e faturamento corporativo.

### Questão 2
Explique a diferença técnica entre um sinal discreto (binário) e um sinal contínuo (analógico), indicando o tipo de dado utilizado no software de TI para armazenar cada um.

### Questão 3
O que é o Duty Cycle de um sinal PWM e por que a técnica PWM é amplamente utilizada em TI/sistemas embarcados para controlar motores e emular saídas analógicas?
